# Lab 2 Solution – Numerical Normalization & Visualization

Dataset: cybersecurity_attacks.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, LabelEncoder


df = pd.read_csv('cybersecurity_attacks.csv')
df.head()

## Select Numerical Features

In [ ]:
numerical_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
num_cols = numerical_cols[:3]
print(num_cols)

## Original Distributions

In [ ]:
for col in num_cols:
    df[col].hist(bins=20)
    plt.title(f'Histogram: {col}')
    plt.show()

    plt.boxplot(df[col].dropna())
    plt.title(f'Boxplot: {col}')
    plt.show()

## Normalization Methods

In [ ]:
X = df[num_cols]

mm = MinMaxScaler()
ss = StandardScaler()
rs = RobustScaler()

X_mm = pd.DataFrame(mm.fit_transform(X), columns=num_cols)
X_ss = pd.DataFrame(ss.fit_transform(X), columns=num_cols)
X_rs = pd.DataFrame(rs.fit_transform(X), columns=num_cols)

# Log transform
X_log = X.copy()
for col in num_cols:
    shift = 1 - X_log[col].min() if X_log[col].min() <= 0 else 0
    X_log[col] = np.log1p(X_log[col] + shift)

print("Scaled samples:")
print(X_mm.head())

## Visual Comparison (First Feature)

In [ ]:
feat = num_cols[0]

datasets = {
    'Original': X[feat],
    'MinMax': X_mm[feat],
    'Standard': X_ss[feat],
    'Robust': X_rs[feat],
    'Log': X_log[feat]
}

for name, series in datasets.items():
    plt.hist(series.dropna(), bins=20)
    plt.title(f'{feat} - {name}')
    plt.show()

## Scatter Plot Colored by Attack Type

In [ ]:
label_candidates = [c for c in df.columns if c.lower() in ['attack type','label','class','target']]
if label_candidates:
    label_col = label_candidates[0]
    y = LabelEncoder().fit_transform(df[label_col].astype(str))

    if len(num_cols) >= 2:
        plt.scatter(df[num_cols[0]], df[num_cols[1]], c=y, s=10)
        plt.xlabel(num_cols[0])
        plt.ylabel(num_cols[1])
        plt.title('Scatter colored by attack type')
        plt.show()
else:
    print("No label column detected")

## Pros & Cons

**Min-Max:** + bounded range | − sensitive to outliers

**Standardization:** + common default | − affected by outliers

**Robust:** + outlier resistant | − unbounded

**Log transform:** + reduces skew | − needs positive data